# EPIC Quickstart

Welcome to **EPIC** (ELIOS Predictive Intelligence Challenge). Unlike a classic competition, EPIC does not hand you a dataset: a **digital twin** (a simulated physical system such as a pump, a motor, or a smart building) runs *live* on the server, and you collect your own training data by listening to its sensor stream in real time. Collecting good data is part of the challenge, exactly as it would be for an engineer instrumenting a real machine. Every contest follows the same **two-phase structure**, designed so that nobody can cheat by submitting "predictions" of data they have already seen:

**Observation phase**: the simulation runs and streams sensor readings to registered participants over a WebSocket. This is your data-collection and model-training window.

**Evaluation phase**: the simulation keeps running, but the stream is closed. The server privately records the *ground truth* for this window; nobody can see it.

**Submission window**: once the evaluation phase ends, you submit a forecast covering every step of the evaluation window. It is scored automatically against the recorded ground truth, and the leaderboard updates.

In this notebook you will log in, find a contest, register, collect live data, build the simplest possible forecast (a *persistence baseline*), submit it, and read your score. Every step explains not just *what* to run but *why*.

## Installation

The cell below installs the official participant SDK ("epic-elios-client"), together with the optional notebook extras (pandas and matplotlib, used here for tables and plots). The SDK wraps the platform's REST API and WebSocket stream so you never have to handle HTTP or reconnection logic yourself. Run it once per environment:

In [ ]:
%pip install "epic-elios-client[notebook]"

## Configure the Client and Log In

**You cannot self-register as a participant**. Accounts are created either by an instructor/administrator, or through a personal **invitation link** the contest organizer sent to your email. If you followed such a link and completed the registration form, the username is your email address and the password is the one you chose on that form.

Logging in returns a **JWT token** that the client should store and attach to every subsequent call. Tokens expire after about an hour; if calls suddenly start failing with authentication errors, simply run this cell again.

Replace the credentials below with your own.

In [ ]:
from epic_client import EPICClient

USER_NAME = "your-username"
PASSWORD = "your-password"

SERVER_URL = "https://epic.elioslab.net"
client = EPICClient(SERVER_URL)

client.login(USER_NAME, PASSWORD)

## Browse the Contests You Can See

After logging in, your next step is to choose a contest to join. The table below shows the contests available to your account: **public contests** (listed for every logged-in user, and anyone may register), plus any **private contests** (visible only to invited users, registered participants, the organizer, and administrators). So if a contest your professor mentioned doesn't appear here, the most likely explanation is that the invitation went to a different email address than the one on your account, ask the organizer to invite the right one. Two columns of the table deserve special attention, because together they define *exactly what you must submit*: **eval_steps** is the number of values you must predict per variable, it equals to prediction_horizon_seconds * sampling_rate_hz (a 60-second horizon sampled at 10Hz means 600 predicted values); and **target_variables** is the subset of sensors that will actually be scored. A contest may stream five sensors but evaluate your forecast on only two of them. You still receive all sensors during the observation phase, the non-target ones are free extra features for your model.

In [ ]:
import pandas as pd

contests = client.list_contests(status="ACTIVE")

rows = []
for c in contests:
    task_cfg = c.get("tasks", [{}])[0].get("configuration", {})
    rows.append({
        "contest_id":                c.get("contest_id"),
        "name":                      c.get("name"),
        "visibility":                c.get("visibility"),
        "twin_id":                   c.get("twin_id"),
        "sampling_rate_hz":          c.get("sampling_rate_hz"),
        "end_of_observation":        c.get("end_of_observation"),
        "prediction_horizon_seconds": c.get("prediction_horizon_seconds"),
        "eval_steps":                task_cfg.get("eval_steps"),
        "target_variables":          ", ".join(task_cfg.get("target_variables", [])),
    })

pd.DataFrame(rows)

## Pick a Contest and Read its Task configuration

Each contest is a different live experiment: a specific digital twin, sensor setup, fault scenario, and forecasting task. Copy the "contest_id" for the contest you want to join and paste it below. The notebook will then read the contest configuration directly from EPIC: how many future steps you must predict ("eval_steps"), which variables count for scoring ("target_variables"), and the sampling rate that turns steps into seconds. Because these values come from the platform, the same notebook can follow any forecasting contest without hard-coded assumptions.

In [ ]:
# Copy a contest_id from the table above
contest_id = "replace-with-contest-id"  

# Read the task configuration from the platform via the SDK helper.
contest = client.get_contest(contest_id)
task_spec = client.get_task_spec(contest_id)
eval_steps = task_spec["eval_steps"]
target_variables = task_spec.get("target_variables") or []
sampling_rate_hz = task_spec["sampling_rate_hz"]

print(f"visibility       : {contest['visibility']}")
print(f"eval_steps       : {eval_steps}")
print(f"target_variables : {target_variables}")
print(f"sampling_rate_hz : {sampling_rate_hz} Hz")
print(f"evaluation window: {eval_steps / sampling_rate_hz:.1f} s")

## Register for the Contest

Before you can receive the live stream or submit predictions, EPIC needs to link your account to this contest. If you already joined (for example through an invitation link), the SDK treats that as success and continues.

In [ ]:
client.register(contest_id)

## Collect Live Observations

This is the moment where the contest becomes a live experiment. The digital twin is already running on the server; the collect() function opens a WebSocket stream and listens to its sensors for the duration you choose. At the same time, it can write the incoming data to CSV, one column per sensor, so your dataset remains available even if you restart the notebook. Every observation is one tick of the simulation. It includes a "sequence_id" (a timestamp), and one numeric reading for each configured sensor. The "sequence_id" is the server-side counter: if you see a gap, your connection missed data, and EPIC does not replay old readings. That is intentional, participants are observing a "live system", not downloading a prepared dataset. Collect while the observation phase is open. After "end_of_observation", the stream closes and the platform starts recording the hidden evaluation window. Joining late simply gives you less history to learn from. Also remember that streamed values are sensor readings, not perfect ground truth: noise, drift, and outliers may be part of the scenario, and handling them is part of the challenge.

In [ ]:
# Adjust to match your contest's observation window
observations = await client.collect(
    contest_id,
    duration_seconds=30,          
    csv_path="epic_observations.csv",
)
print(f"Collected {len(observations)} observations")
observations[:2]

## Build a DataFrame

The stream gives you one observation at a time: metadata such as "sequence_id" and "timestamp", plus the actual measurements inside a nested "sensors" dictionary. For analysis and modelling, it is easier to turn that nested structure into a table where each row is one time step and each sensor has its own column. Keep all sensor columns, not only the variables that will be scored. Target variables define what you must predict, but the other sensors can still be useful signals for your model.

In [ ]:
rows = []
for obs in observations:
    rows.append({
        "sequence_id": obs["sequence_id"],
        "timestamp":   obs["timestamp"],
        **obs["sensors"],
    })

df = pd.DataFrame(rows)
df.head()

## Plot a Sensor Channel

Always look at your data before modeling it. The plot below shows the first sensor channel against "sequence_id"; with a mechanical twin you should see an oscillation, with a thermal one a slow trend. Jagged spikes are usually configured sensor noise or outliers (information about how hard the organizer made this contest).

In [ ]:
import matplotlib.pyplot as plt

sensor_columns = [c for c in df.columns if c not in {"sequence_id", "timestamp"}]
sensor_name = sensor_columns[0]

df.plot(x="sequence_id", y=sensor_name, figsize=(10, 4), title=f"{sensor_name} over time")
plt.xlabel("Sequence ID")
plt.ylabel(sensor_name)
plt.tight_layout()
plt.show()

## Build the Forecast

At this point you have a short history of sensor readings. The forecasting task is to use that history to predict the target variables over the hidden evaluation window. In a real submission, you could approach this in many ways: fit a physics-inspired model of the system, train a regression model on lagged sensor values, use classical time-series methods, or build a neural model such as an LSTM, GRU, temporal convolution, or transformer. The important rule is that your model may use the streamed sensor data, but it must produce future values for the configured target variables.

This notebook uses the simplest possible baseline so the full EPIC workflow stays easy to follow. A *persistence* forecast assumes that each future value stays equal to the last value observed. It is not sophisticated, but it creates a valid submission and gives you a first score to improve on with better models. The payload must include every target variable, and each target must contain exactly "eval_steps" numbers. The cell below builds only those required target series; any extra sensors can help your model, but they are not scored directly.

In [ ]:
# Get the list of target variables from the task specification; 
forecast_targets = target_variables or sensor_columns

# Verify that all target variables are present in the collected data.
missing_targets = [target for target in forecast_targets if target not in df.columns]
if missing_targets:
    raise ValueError(f"Target variables not present in collected data: {missing_targets}")

# Get the last observed value for each target variable.
last_values = df[forecast_targets].iloc[-1]

# Persist the last observed value for every target variable and every step.
payload = {
    "forecast": {
        target: [float(last_values[target])] * eval_steps
        for target in forecast_targets
    }
}

print(f"Forecast shape: {len(forecast_targets)} target variable(s) x {eval_steps} steps")
print(payload)

## Submit your forecast

Submissions are only accepted **after the evaluation phase has fully elapsed**, that is, after end_of_observation + prediction_horizon_seconds. This is the platform's anti-cheating guarantee: by the time anyone may submit, the ground truth already exists on the server and nobody could have seen it, so every accepted forecast is genuinely prospective. Submitting earlier returns a clear "409" error telling you when the window opens; just wait and re-run this cell. The response shows your submission in "pending" status, scoring happens asynchronously, usually within a second or two.

In [ ]:
submission = client.submit(
    contest_id = contest_id,
    task_id = "forecasting",
    payload = payload,
)

print("Submission response:", submission)

## Check score and the Leaderboard

A submission moves from "pending" to "evaluated" or "failed" (with the reason recorded, typically a malformed payload). The default metric is the Mean Absolute Error (MAE) in the sensor's own units, where *lower is better*. By default you are scored against the clean **ground truth**, the noiseless latent state of the twin, so sensor noise does not unfairly penalize a good model. The leaderboard keeps each participant's **best submission**, so you can resubmit as often as you like: a worse attempt never hurts your ranking. If your submission shows "pending", wait a moment and re-run.

In [ ]:
scores = client.get_scores(CONTEST_ID)
leaderboard = client.get_leaderboard(CONTEST_ID)

print("=== My Submissions ===")
for sub in scores.get("submissions", []):
    print(f"  {sub['submission_id'][:8]}…  status={sub['status']}  "
          f"scores={[round(s['value'], 4) for s in sub.get('scores', [])]}")

print("\n=== Leaderboard ===")
for entry in leaderboard.get("entries", []):
    print(f"  Rank {entry['rank']}  user={entry['user_id'][:8]}…  score={entry['score']:.4f}")

## Next Steps

The persistence baseline is a starting point, not a strategy. Natural improvements, roughly in order of effort: exploit the physics of the twin, use classical time-series models (ARIMA) on each target, or train a learned regressor using *all* streamed sensors as features for the target variables. Whatever you build, the loop stays the same: collect during observation, predict "eval_steps" values per target, submit when the window opens, and let the leaderboard keep your best!